# Sistemas não lineares e otimização

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/05-otimizacao.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. aplicar o método de Newton a um sistema de equações não lineares;
> 2. reconhecer que encontrar um equilíbrio e minimizar uma energia são o mesmo problema;
> 3. comparar descida pelo gradiente, Newton e quase-Newton (BFGS) quanto a custo, velocidade e robustez;
> 4. verificar se um ponto é de fato um mínimo, e não um ponto de sela;
> 5. tratar restrições com multiplicadores de Lagrange e usar `scipy.optimize`.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from scipy.optimize import minimize
from cnlivro.otimizacao import newton_sistema, gradiente_descendente, bfgs, rosenbrock, grad_rosenbrock

## 1. Problema de engenharia: um bloco pendurado em duas molas

Um bloco de massa $m = 5$ kg está preso a duas molas, ancoradas no teto nos pontos $A_1 = (0, 0)$ e $A_2 = (2, 0)$, em metros. As molas têm o mesmo comprimento natural, $L = 1{,}5$ m, e rigidezes diferentes, $k_1 = 150$ N/m e $k_2 = 250$ N/m. Onde o bloco fica parado?

Há duas formas de escrever a resposta:

- **Equilíbrio de forças.** A força resultante no bloco é zero: duas equações não lineares, uma para cada direção, nas incógnitas $(x, y)$.
- **Mínimo de energia.** O bloco para na posição que minimiza a energia potencial total, elástica mais gravitacional.

As duas formas dão a mesma resposta, e isso não é coincidência. O princípio da energia potencial mínima está por trás do cálculo de estruturas, de cabos e do método dos elementos finitos. Este capítulo usa o bloco para mostrar os dois lados, os **sistemas não lineares** e a **otimização**.

A energia potencial do bloco na posição $\mathbf{p} = (x, y)$ é

$$
E(\mathbf{p}) = \tfrac12 k_1 \big(\ell_1(\mathbf{p}) - L\big)^2 + \tfrac12 k_2 \big(\ell_2(\mathbf{p}) - L\big)^2 + m g\, y,
\qquad \ell_i(\mathbf{p}) = \|\mathbf{p} - A_i\| .
$$ (eq-energia-molas)

In [ ]:
A1, A2 = np.array([0.0, 0.0]), np.array([2.0, 0.0])
k1, k2, L, m, g = 150.0, 250.0, 1.5, 5.0, 9.81

def energia(p):
    l1, l2 = np.linalg.norm(p - A1), np.linalg.norm(p - A2)
    return 0.5 * k1 * (l1 - L) ** 2 + 0.5 * k2 * (l2 - L) ** 2 + m * g * p[1]

def gradiente(p):
    # ∇E = −(força resultante sobre o bloco)
    d1, d2 = p - A1, p - A2
    l1, l2 = np.linalg.norm(d1), np.linalg.norm(d2)
    return k1 * (l1 - L) * d1 / l1 + k2 * (l2 - L) * d2 / l2 + np.array([0.0, m * g])

def hessiana(p):
    H = np.zeros((2, 2))
    for A, k in [(A1, k1), (A2, k2)]:
        d = p - A
        l = np.linalg.norm(d)
        u = np.outer(d, d) / l**2
        H += k * (u + (1 - L / l) * (np.eye(2) - u))
    return H

## 2. A ideia em uma imagem

As curvas de nível de $E$ formam um "mapa do relevo". Em cada ponto, $-\nabla E$ é a **força resultante** sobre o bloco e aponta morro abaixo. O equilíbrio estável é o fundo do vale. A figura mostra três maneiras de chegar lá a partir do mesmo ponto.

In [ ]:
def malha_energia(x=np.linspace(-0.6, 2.6, 161), y=np.linspace(-2.2, 1.0, 161)):
    X, Y = np.meshgrid(x, y)
    Z = np.array([[energia(np.array([a, b])) for a in x] for b in y])
    return x, y, Z

xg, yg, Zg = malha_energia()
inicio = np.array([0.2, -0.3])
caminhos = {
    "descida pelo gradiente": (gradiente_descendente(energia, gradiente, inicio, tol=1e-6), CORES["erro"]),
    "BFGS": (bfgs(energia, gradiente, inicio, tol=1e-6), CORES["aproximacao"]),
    "Newton": (newton_sistema(gradiente, hessiana, inicio), CORES["destaque"]),
}
fig = go.Figure(go.Contour(x=xg, y=yg, z=Zg, contours=dict(start=-60, end=120, size=10, coloring="lines"),
                           colorscale=[[0, "#bbbbbb"], [1, "#bbbbbb"]], showscale=False, hoverinfo="skip",
                           line=dict(width=1)))
for nome, (r, cor) in caminhos.items():
    h = np.array(r.historico)
    fig.add_trace(go.Scatter(x=h[:, 0], y=h[:, 1], mode="lines+markers", name=f"{nome} ({r.iteracoes} passos)",
                             line=dict(color=cor), marker=dict(size=6)))
fig.add_trace(go.Scatter(x=[A1[0], A2[0]], y=[A1[1], A2[1]], mode="markers", name="ancoragens",
                         marker=dict(symbol="triangle-down", size=14, color=CORES["exata"])))
fig.add_trace(go.Scatter(x=[inicio[0]], y=[inicio[1]], mode="markers", name="início",
                         marker=dict(size=11, color=CORES["exata"], symbol="circle-open", line=dict(width=2))))
fig.update_xaxes(title="x (m)", range=[-0.6, 2.6])
fig.update_yaxes(title="y (m)", range=[-2.2, 1.0], scaleanchor="x")
fig.update_layout(title="O equilíbrio é o fundo do vale de energia", height=560)
mostrar(fig)
p_eq = caminhos["BFGS"][0].x
print(f"equilíbrio: x = {numero(p_eq[0], 4)} m, y = {numero(p_eq[1], 4)} m;  E = {numero(energia(p_eq), 2)} J")

**O que notar:** a descida pelo gradiente só enxerga a inclinação local: anda na direção de maior declive, passa do ponto, corrige, e por isso dá mais passos. Newton usa também a curvatura e vai quase em linha reta. O BFGS aprende a curvatura no caminho e fica entre os dois. Todos chegam ao mesmo ponto, um pouco à direita do meio, porque a mola da direita é mais rígida.

## 3. Formulação matemática

### 3.1 Newton para sistemas

Queremos $\mathbf{F}(\mathbf{x}) = \mathbf{0}$, com $\mathbf{F}: \mathbb{R}^n \to \mathbb{R}^n$. A ideia é a mesma do capítulo 3: trocar $\mathbf{F}$ pela sua aproximação linear em torno do ponto atual,

$$
\mathbf{F}(\mathbf{x}_k + \Delta\mathbf{x}) \approx \mathbf{F}(\mathbf{x}_k) + J(\mathbf{x}_k)\,\Delta\mathbf{x},
\qquad J_{ij} = \frac{\partial F_i}{\partial x_j},
$$

e anular a aproximação. Cada passo resolve um **sistema linear** (capítulo 4):

$$
J(\mathbf{x}_k)\,\Delta\mathbf{x} = -\mathbf{F}(\mathbf{x}_k), \qquad \mathbf{x}_{k+1} = \mathbf{x}_k + \Delta\mathbf{x}.
$$ (eq-newton-sistema)

Perto de uma raiz com $J$ invertível, a convergência é **quadrática**, como em uma variável. O preço é calcular as $n^2$ derivadas de $J$ e resolver um sistema $n \times n$ por passo. As derivadas podem vir de fórmulas, de diferenças finitas (capítulo 10) ou de diferenciação automática (capítulo 9).

### 3.2 Equilíbrio é mínimo de energia

A força resultante sobre o bloco é $\mathbf{f} = -\nabla E$. Então o equilíbrio de forças, $\mathbf{f} = \mathbf{0}$, é a mesma coisa que $\nabla E = \mathbf{0}$: um sistema não linear com $\mathbf{F} = \nabla E$ e jacobiana igual à **hessiana** $H = \nabla^2 E$.

Mas $\nabla E = \mathbf{0}$ só diz que o ponto é **estacionário**. O tipo de ponto depende da hessiana:

| Hessiana em $\mathbf{x}^*$ | Tipo de ponto | Equilíbrio |
| --- | --- | --- |
| positiva definida (autovalores $> 0$) | mínimo local | **estável** |
| indefinida (autovalores de sinais opostos) | ponto de sela | instável |
| negativa definida | máximo local | instável |

Esta é a **condição de segunda ordem**. Ela é a diferença entre um bloco parado e um lápis equilibrado na ponta.

### 3.3 Descida pelo gradiente

O caminho mais simples para um mínimo é andar contra o gradiente:

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha_k \nabla f(\mathbf{x}_k).
$$

O passo $\alpha_k$ pode ser fixo, mas é melhor escolhê-lo por **busca linear**: começa com $\alpha = 1$ e divide por 2 até que $f$ diminua o suficiente (condição de Armijo, $f(\mathbf{x} + \alpha\mathbf{d}) \le f(\mathbf{x}) + c\,\alpha\,\nabla f^{\mathsf T}\mathbf{d}$, com $c$ pequeno).

A velocidade depende da forma do vale. Perto do mínimo, $f$ é quase uma quadrática com hessiana $H$. Se os autovalores de $H$ vão de $\lambda_{\min}$ a $\lambda_{\max}$, o número de condição $\kappa = \lambda_{\max}/\lambda_{\min}$ mede quão alongado é o vale, e o erro cai, no melhor caso, por um fator

$$
\frac{\kappa - 1}{\kappa + 1}
$$ (eq-taxa-gradiente)

por passo. Com $\kappa = 100$, o fator é $0{,}98$: são cerca de 115 passos para ganhar um algarismo. Num vale alongado, o gradiente aponta quase perpendicular ao fundo do vale, e o método anda em ziguezague. A seção 5 mostra isso.

### 3.4 Newton para otimização

Aplicar Newton {eq}`eq-newton-sistema` a $\nabla f = \mathbf{0}$ dá

$$
H(\mathbf{x}_k)\,\Delta\mathbf{x} = -\nabla f(\mathbf{x}_k).
$$

Isso equivale a aproximar $f$ por uma quadrática (o polinômio de Taylor de grau 2) e pular direto para o ponto estacionário dela. A convergência é quadrática e não depende de $\kappa$. Mas há dois problemas:

1. É preciso a hessiana, com $n^2$ derivadas segundas, e resolver um sistema por passo.
2. Newton procura **pontos estacionários**, não mínimos. Se $H$ não é positiva definida, o passo pode subir o morro, e o método pode convergir para uma sela ou para um máximo. A seção 6 mostra esse problema no bloco.

### 3.5 Quase-Newton: BFGS

Os métodos **quase-Newton** constroem uma aproximação da hessiana usando só gradientes. Depois de um passo $\mathbf{s}_k = \mathbf{x}_{k+1} - \mathbf{x}_k$, a variação do gradiente $\mathbf{y}_k = \nabla f_{k+1} - \nabla f_k$ informa a curvatura ao longo de $\mathbf{s}_k$. A nova aproximação $B_{k+1}$ é escolhida para satisfazer a **equação secante**

$$
B_{k+1}\,\mathbf{s}_k = \mathbf{y}_k,
$$

a versão multidimensional da secante do capítulo 3. O BFGS (Broyden, Fletcher, Goldfarb e Shanno) faz a menor correção possível que satisfaz essa equação e mantém $B$ **positiva definida**. Por isso a direção $-B^{-1}\nabla f$ sempre desce, e, com busca linear, $f$ diminui a cada passo. A convergência é superlinear, e o BFGS é o método padrão de `scipy.optimize.minimize` quando não há restrições.

| Método | Precisa de | Custo por passo | Convergência | Desce sempre? |
| --- | --- | --- | --- | --- |
| Gradiente | $\nabla f$ | $O(n)$ | linear, taxa $(\kappa-1)/(\kappa+1)$ | sim (com busca linear) |
| Newton | $\nabla f$, $H$ | $O(n^3)$ | quadrática | **não** |
| BFGS | $\nabla f$ | $O(n^2)$ | superlinear | sim (com busca linear) |

### 3.6 Restrições e multiplicadores de Lagrange

Muitos problemas têm restrições: um bloco não atravessa o chão, uma viga tem altura máxima, uma vazão não pode ser negativa. Para minimizar $f(\mathbf{x})$ sujeito a $c(\mathbf{x}) \ge 0$, há duas possibilidades no ótimo:

- a restrição está **inativa**, $c(\mathbf{x}^*) > 0$, e vale $\nabla f = \mathbf{0}$ como antes;
- a restrição está **ativa**, $c(\mathbf{x}^*) = 0$, e no ótimo os gradientes ficam paralelos:

$$
\nabla f(\mathbf{x}^*) = \lambda\, \nabla c(\mathbf{x}^*), \qquad \lambda \ge 0 .
$$ (eq-lagrange)

O número $\lambda$ é o **multiplicador de Lagrange**. Ele mede quanto o ótimo melhoraria se a restrição fosse afrouxada um pouco, e muitas vezes tem significado físico. No bloco apoiado no chão, $\lambda$ é a **força normal** que o chão faz (seção 7.3). As condições {eq}`eq-lagrange` com várias restrições são as condições de Karush-Kuhn-Tucker (KKT). Na prática, usamos bibliotecas que as resolvem, como `scipy.optimize.minimize` com `method="SLSQP"` ou `"trust-constr"`.

## 4. À mão

Ache a interseção do círculo $x^2 + y^2 = 4$ com a hipérbole $xy = 1$, perto de $(2;\ 0{,}5)$. O sistema é

$$
\mathbf{F}(x, y) = \begin{bmatrix} x^2 + y^2 - 4 \\ xy - 1 \end{bmatrix},
\qquad
J(x, y) = \begin{bmatrix} 2x & 2y \\ y & x \end{bmatrix}.
$$

No ponto inicial, $\mathbf{F} = (0{,}25;\ 0)$ e $J = \begin{bmatrix} 4 & 1 \\ 0{,}5 & 2 \end{bmatrix}$, com $\det J = 7{,}5$. O sistema $J\,\Delta\mathbf{x} = -\mathbf{F}$ dá, pela regra de Cramer,

$$
\Delta x = \frac{(-0{,}25)(2) - (1)(0)}{7{,}5} = -0{,}06667,
\qquad
\Delta y = \frac{(4)(0) - (0{,}5)(-0{,}25)}{7{,}5} = 0{,}01667 .
$$

O novo ponto é $(1{,}93333;\ 0{,}51667)$. A solução exata é $x = \sqrt{2 + \sqrt 3} = 1{,}93185$, $y = 1/x = 0{,}51764$: depois de um passo, o erro já é da ordem de $10^{-3}$. O computador continua, e o erro mostra a convergência quadrática, com o número de algarismos corretos dobrando a cada passo:

In [ ]:
F = lambda v: np.array([v[0]**2 + v[1]**2 - 4.0, v[0] * v[1] - 1.0])
J = lambda v: np.array([[2 * v[0], 2 * v[1]], [v[1], v[0]]])
exata = np.array([np.sqrt(2 + np.sqrt(3)), np.sqrt(2 - np.sqrt(3))])

r = newton_sistema(F, J, [2.0, 0.5], tol=1e-15)
for k, xk in enumerate(r.historico):
    print(f"k = {k}:  x = {xk[0]:.12f}  y = {xk[1]:.12f}   erro = {np.linalg.norm(xk - exata):.1e}")

## 5. Explore

### 5.1 O ziguezague num vale alongado

A função $f(x, y) = \tfrac12 (x^2 + \kappa\, y^2)$ é um vale cujas curvas de nível são elipses, com hessiana de autovalores $1$ e $\kappa$. Arraste o slider para alongar o vale e veja a descida pelo gradiente, com o melhor passo possível em cada iteração, partindo do mesmo ponto.

In [ ]:
def figura_ziguezague(kappa, tol=1e-6):
    Q = np.diag([1.0, kappa])
    x = np.array([10.0, 1.0])
    caminho = [x.copy()]
    while np.linalg.norm(Q @ x) > tol and len(caminho) < 2000:
        gk = Q @ x
        alfa = gk @ gk / (gk @ Q @ gk)        # passo ótimo numa quadrática (busca linear exata)
        x = x - alfa * gk
        caminho.append(x.copy())
    c = np.array(caminho)
    t = np.linspace(0, 2 * np.pi, 200)
    fig = go.Figure()
    for nivel in [0.5, 2, 8, 20, 50]:
        fig.add_trace(go.Scatter(x=np.sqrt(2 * nivel) * np.cos(t), y=np.sqrt(2 * nivel / kappa) * np.sin(t), mode="lines",
                                 line=dict(color=CORES["referencia"], width=1), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=c[:, 0], y=c[:, 1], mode="lines+markers", name="descida pelo gradiente",
                             line=dict(color=CORES["erro"]), marker=dict(size=5)))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", name="mínimo", marker=dict(color=CORES["destaque"], size=10)))
    fig.update_xaxes(title="x", range=[-1, 11])
    fig.update_yaxes(title="y", range=[-1.6, 1.6])
    fig.update_layout(title=f"κ = {kappa}: {len(caminho) - 1} iterações até ‖∇f‖ < 10⁻⁶", height=380)
    return fig

valores_kappa = [1, 2, 5, 10, 20, 50, 100]
quadros = [(str(k), figura_ziguezague(k)) for k in valores_kappa]
mostrar(com_controle(quadros, rotulo="κ = ", ativo=3, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com $\kappa = 1$ as curvas de nível são círculos, o gradiente aponta direto para o mínimo e basta um passo. Quando $\kappa$ cresce, o gradiente aponta para a parede do vale, e o método ziguezagueia. O número de iterações cresce aproximadamente como $\kappa$, como prevê {eq}`eq-taxa-gradiente`. Newton resolveria qualquer um destes casos em **um** passo, porque usa a hessiana e "desfaz" o alongamento.

### 5.2 Quantas iterações cada método gasta

Na energia do bloco, comparamos a norma do gradiente, isto é, a força resultante, ao longo das iterações dos três métodos da figura da seção 2:

In [ ]:
resultados = {nome: r for nome, (r, cor) in caminhos.items()}
resultados["Newton"].erros = [np.linalg.norm(gradiente(x)) for x in resultados["Newton"].historico]
cor_metodo = {"descida pelo gradiente": CORES["erro"], "gradiente": CORES["erro"], "BFGS": CORES["aproximacao"],
              "Newton": CORES["destaque"]}
fig = grafico_iteracoes(resultados, titulo="Força resultante ‖∇E‖ ao longo das iterações")
for tr in fig.data:
    tr.update(line_color=cor_metodo[tr.name], marker_color=cor_metodo[tr.name])
fig.update_yaxes(title="‖∇E‖ (N)")
mostrar(fig)

**O que notar:** num gráfico semilog, a descida pelo gradiente é uma reta, a marca da convergência **linear**. Newton e BFGS fazem curvas para baixo, a marca da convergência **superlinear**: cada passo ganha mais algarismos que o anterior.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Escreva em Python a função `equilibrio(x0=None)` que encontra a posição de equilíbrio do bloco preso às duas molas, resolvendo $\nabla E = \mathbf{0}$ pelo método de Newton. Use o gradiente e a hessiana analíticos que já tenho (`gradiente` e `hessiana`). Pare quando o passo relativo for menor que $10^{-12}$ e lance um erro se não convergir em 50 iterações. Teste verificando que a força resultante no resultado é zero."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
def equilibrio(x0=None, tol=1e-12, max_iter=50):
    if x0 is None:
        x0 = (A1 + A2) / 2               # chute: bloco no meio, entre as ancoragens
    x = np.array(x0, dtype=float)
    for k in range(max_iter):
        dx = np.linalg.solve(hessiana(x), -gradiente(x))
        x = x + dx
        if np.linalg.norm(dx) < tol * np.linalg.norm(x):
            return x, k + 1
    raise RuntimeError("Newton não convergiu")

# Teste escrito pelo agente
p, iteracoes = equilibrio()
forca = -gradiente(p)
print(f"posição: ({p[0]:.4f}, {p[1]:.4f}) m em {iteracoes} iterações")
print(f"força resultante: {np.linalg.norm(forca):.1e} N  ->  teste {'passou' if np.linalg.norm(forca) < 1e-8 else 'falhou'}")

O código é exatamente o método pedido, converge em poucas iterações, e o teste passa: a força resultante é praticamente zero. Mas compare com a figura da seção 2.

### 6.2 Verificação: o resultado faz sentido físico?

O bloco "em equilíbrio" está em $y \approx +0{,}30$ m, **acima** da linha das ancoragens. As molas estão **comprimidas** e empurram o bloco para cima, equilibrando o peso. As forças de fato se anulam. Mas esse equilíbrio se mantém? A condição de segunda ordem da seção 3.2 responde:

In [ ]:
autovalores, autovetores = np.linalg.eigh(hessiana(p))
print("autovalores da hessiana:", autovalores.round(1), "N/m")
print("direção do autovalor negativo:", autovetores[:, 0].round(3))
print(f"energia no ponto do agente: {numero(energia(p), 1)} J;   no equilíbrio da seção 2: {numero(energia(p_eq), 1)} J")

**O que notar:** um autovalor é **negativo**, e a direção dele é quase vertical. O ponto é uma **sela**: na horizontal a energia tem um mínimo, mas na vertical tem um máximo. Se o bloco descer um milímetro, as molas ficam mais comprimidas e mais deitadas, empurram menos para cima, e o peso vence: o bloco atravessa a linha das ancoragens e despenca até ficar pendurado. É o fenômeno de *snap-through* das treliças rasas, o mesmo "estalo" da tampa de um pote. A energia na sela é de cerca de $+54$ J, contra $-60$ J no equilíbrio verdadeiro.

O agente não errou na fórmula, e Newton funcionou exatamente como devia: encontrou um ponto onde $\nabla E = \mathbf{0}$. O erro é de **método**: Newton procura pontos estacionários, e o chute "natural", com o bloco entre as ancoragens e as molas comprimidas, está na bacia de atração da sela. O teste do resíduo não tinha como ver isso.

### 6.3 Para onde vai Newton

Repetimos o código do agente a partir de uma grade de chutes iniciais e pintamos cada chute com a cor do ponto a que ele leva:

In [ ]:
def classifica(metodo, xs, ys):
    # 0 = mínimo (equilíbrio estável), 1 = sela, 2 = não convergiu
    mapa = np.full((len(ys), len(xs)), 2.0)
    with np.errstate(all="ignore"):         # alguns chutes caem em cima de uma ancoragem
        _classifica(metodo, xs, ys, mapa)
    return mapa

def _classifica(metodo, xs, ys, mapa):
    for i, b in enumerate(ys):
        for j, a in enumerate(xs):
            try:
                x = metodo(np.array([a, b]))
            except Exception:
                continue
            if x is not None and np.all(np.isfinite(x)) and np.linalg.norm(gradiente(x)) < 1e-5:
                mapa[i, j] = 0.0 if np.all(np.linalg.eigvalsh(hessiana(x)) > 0) else 1.0

def por_newton(x0):
    return equilibrio(x0)[0]

def por_bfgs(x0):
    r = bfgs(energia, gradiente, x0, tol=1e-6)
    return r.x if r.convergiu else None

xs = np.linspace(-0.6, 2.6, 81)
ys = np.linspace(-2.2, 1.0, 81)
cores_bacia = [[0, CORES["destaque"]], [0.33, CORES["destaque"]], [0.34, CORES["erro"]], [0.66, CORES["erro"]],
               [0.67, "#dddddd"], [1, "#dddddd"]]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Newton (código do agente)", "BFGS com busca linear"),
                    horizontal_spacing=0.08)
for col, metodo in [(1, por_newton), (2, por_bfgs)]:
    fig.add_trace(go.Heatmap(x=xs, y=ys, z=classifica(metodo, xs, ys), zmin=0, zmax=2, colorscale=cores_bacia,
                             showscale=False, hovertemplate="chute (%{x:.2f}, %{y:.2f})<extra></extra>"), 1, col)
    fig.add_trace(go.Scatter(x=[A1[0], A2[0]], y=[0, 0], mode="markers", showlegend=False,
                             marker=dict(symbol="triangle-down", size=12, color=CORES["exata"])), 1, col)
    fig.add_trace(go.Scatter(x=[1.0], y=[0.0], mode="markers", showlegend=False,
                             marker=dict(size=10, color="white", line=dict(color=CORES["exata"], width=2))), 1, col)
for nome, cor in [("leva ao mínimo (estável)", CORES["destaque"]), ("leva à sela (instável)", CORES["erro"]),
                  ("não converge", "#dddddd")]:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=nome, marker=dict(size=12, color=cor, symbol="square")))
fig.update_xaxes(title="x₀ (m)")
fig.update_yaxes(title="y₀ (m)", col=1)
fig.update_layout(title="Para onde cada chute inicial leva", height=460, legend=dict(orientation="h", y=-0.2))
mostrar(fig)

**O que notar:** à esquerda, há uma região grande, em vermelho, de chutes que levam Newton à sela. O círculo branco é o chute do agente, o ponto médio entre as ancoragens, bem no meio dessa região. À direita, o BFGS com busca linear diminui a energia a cada passo e, por isso, não para numa sela: de praticamente todos os chutes, ele chega ao equilíbrio estável.

### 6.4 A correção

1. **Verificar a condição de segunda ordem.** Todo resultado de "equilíbrio" ou de "mínimo" deve vir acompanhado dos autovalores da hessiana. Um autovalor negativo é um alarme.
2. **Usar um método que desce.** Para minimizar, use BFGS (ou Newton com busca linear e proteção contra hessianas indefinidas). Na prática, `scipy.optimize.minimize`.
3. **Comparar com a biblioteca**, que fecha a lista de verificação do capítulo 0:

In [ ]:
ref = minimize(energia, x0=(A1 + A2) / 2, jac=gradiente, method="BFGS", options={"gtol": 1e-10})
print("scipy BFGS: ", ref.x.round(6), " E =", round(ref.fun, 4), "J")
print("cnlivro BFGS:", p_eq.round(6), " E =", round(energia(p_eq), 4), "J")
compara(p_eq, ref.x, rtol=1e-6)
print("autovalores da hessiana no resultado:", np.linalg.eigvalsh(hessiana(ref.x)).round(1), "N/m (todos positivos: estável)")

> **Lição.** Resíduo zero prova que o ponto é estacionário, não que é o ponto que você quer. Em otimização e em equilíbrio, a verificação inclui a **condição de segunda ordem** e um **teste físico**: o bloco pendurado deveria estar abaixo das ancoragens.

## 7. Quando falha

### 7.1 O vale estreito de Rosenbrock

A função de Rosenbrock, $f(x, y) = (1 - x)^2 + 100\,(y - x^2)^2$, é um teste clássico: o mínimo, em $(1, 1)$, fica no fundo de um vale estreito e curvo. Partindo de $(-1{,}2;\ 1)$:

In [ ]:
def hess_rosenbrock(v):
    x, y = v
    return np.array([[2 - 400 * (y - x**2) + 800 * x**2, -400 * x], [-400 * x, 200.0]])

x0 = np.array([-1.2, 1.0])
r_gd = gradiente_descendente(rosenbrock, grad_rosenbrock, x0, tol=1e-6, max_iter=20_000)
r_bfgs = bfgs(rosenbrock, grad_rosenbrock, x0, tol=1e-6)
r_newton = newton_sistema(grad_rosenbrock, hess_rosenbrock, x0, tol=1e-12)
r_newton.erros = [np.linalg.norm(grad_rosenbrock(x)) for x in r_newton.historico]
for nome, r in [("gradiente", r_gd), ("BFGS", r_bfgs), ("Newton", r_newton)]:
    print(f"{nome:10s}: {r.iteracoes:6d} iterações, x = {np.round(r.x, 6)}, {'convergiu' if r.convergiu else 'NÃO convergiu: ' + r.motivo}")
fig = grafico_iteracoes({"gradiente": r_gd, "BFGS": r_bfgs, "Newton": r_newton}, titulo="Rosenbrock: ‖∇f‖ ao longo das iterações")
for tr in fig.data:
    tr.update(line_color=cor_metodo[tr.name], marker_color=cor_metodo[tr.name])
fig.update_xaxes(type="log", title="iteração (escala log)")
fig.update_yaxes(title="‖∇f‖")
mostrar(fig)

**O que notar:** a descida pelo gradiente precisa de milhares de iterações para atravessar o vale, que tem $\kappa$ da ordem de $10^3$ perto do mínimo. O BFGS resolve em poucas dezenas, e Newton, com a hessiana exata, em menos ainda. Num problema com milhões de variáveis, como no treinamento de redes neurais (capítulo 9), a hessiana é cara demais, e variantes da descida pelo gradiente voltam a ser a escolha.

### 7.2 Mínimos locais

Os métodos deste capítulo são **locais**: encontram o mínimo do vale onde começam. A função de Himmelblau, $f(x, y) = (x^2 + y - 11)^2 + (x + y^2 - 7)^2$, tem quatro mínimos. O BFGS, partindo de chutes diferentes, encontra mínimos diferentes:

In [ ]:
himmelblau = lambda v: (v[0]**2 + v[1] - 11)**2 + (v[0] + v[1]**2 - 7)**2
grad_himmelblau = lambda v: np.array([4 * v[0] * (v[0]**2 + v[1] - 11) + 2 * (v[0] + v[1]**2 - 7),
                                      2 * (v[0]**2 + v[1] - 11) + 4 * v[1] * (v[0] + v[1]**2 - 7)])
for chute in [(0, 0), (-1, 1), (-2, -2), (2, -2)]:
    r = bfgs(himmelblau, grad_himmelblau, chute)
    print(f"chute {str(chute):9s} -> mínimo em ({r.x[0]: .4f}, {r.x[1]: .4f}),  f = {himmelblau(r.x):.1e}")

Aqui os quatro mínimos têm o mesmo valor, $f = 0$. Em problemas reais, em geral não têm, e um método local pode entregar um mínimo ruim sem aviso. As defesas são partir de vários chutes (*multistart*), usar métodos globais, como `scipy.optimize.differential_evolution`, e usar o conhecimento do problema para escolher o chute.

### 7.3 Esquecer uma restrição

Suponha que o chão esteja em $y = -1{,}2$ m. O equilíbrio da seção 2, em $y \approx -1{,}33$ m, fica **abaixo do chão**: o modelo sem restrição dá uma resposta fisicamente impossível, e nenhum teste numérico acusa isso. Com a restrição $c(\mathbf{p}) = y + 1{,}2 \ge 0$:

In [ ]:
chao = -1.2
restricao = {"type": "ineq", "fun": lambda p: p[1] - chao, "jac": lambda p: np.array([0.0, 1.0])}
res = minimize(energia, x0=np.array([1.0, -0.5]), jac=gradiente, method="SLSQP", constraints=[restricao],
               options={"ftol": 1e-12})
p_c = res.x
grad_c = gradiente(p_c)
print(f"com o chão: x = {p_c[0]:.4f} m, y = {p_c[1]:.4f} m  (restrição ativa: y = {chao})")
print(f"∇E no ótimo: ({grad_c[0]:.1e}, {grad_c[1]:.2f}) N   -> paralelo a ∇c = (0, 1), como pede a equação de Lagrange")
print(f"multiplicador λ = ∂E/∂y = {grad_c[1]:.2f} N = força normal do chão sobre o bloco")

**O que notar:** no ótimo com restrição, $\partial E/\partial x = 0$, mas $\partial E/\partial y \ne 0$: o gradiente aponta na direção de $\nabla c = (0, 1)$, como manda {eq}`eq-lagrange`. O multiplicador $\lambda \approx 31$ N é a força que o chão faz para cima, ou seja, a parte do peso, $mg \approx 49$ N, que as molas não sustentam nessa posição. O multiplicador de Lagrange é uma grandeza física que se pode medir.

### 7.4 Gradiente errado

Todos os métodos deste capítulo dependem do gradiente. Um gradiente com um erro de fórmula faz o método convergir devagar, parar no lugar errado ou não convergir. Antes de otimizar, compare o gradiente analítico com diferenças finitas em alguns pontos aleatórios (`scipy.optimize.check_grad` faz isso). É uma verificação barata, e um agente deveria escrevê-la sem que você peça:

In [ ]:
from scipy.optimize import check_grad
rng = np.random.default_rng(1)
pontos = rng.uniform([-0.5, -2.0], [2.5, 0.8], size=(5, 2))
print("diferença entre gradiente analítico e numérico:", [f"{check_grad(energia, gradiente, q):.1e}" for q in pontos])

**O que notar:** as diferenças são da ordem de $10^{-6}$ N, o erro esperado das diferenças finitas (capítulo 2), contra gradientes da ordem de 10 a 100 N. Um erro de fórmula daria diferenças do tamanho do próprio gradiente.

## Resumo

| Ideia | Em uma frase |
| --- | --- |
| Newton para sistemas | cada passo resolve $J\,\Delta\mathbf{x} = -\mathbf{F}$; convergência quadrática perto da raiz |
| Equilíbrio = mínimo | $\mathbf{f} = -\nabla E$; equilíbrio estável é mínimo local da energia |
| Condição de 2ª ordem | hessiana positiva definida: mínimo; indefinida: sela |
| Descida pelo gradiente | barata por passo, mas lenta em vales alongados: taxa $(\kappa-1)/(\kappa+1)$ |
| Newton para otimização | rápido, mas precisa da hessiana e pode convergir para selas |
| BFGS | aprende a curvatura com gradientes; desce sempre; é o padrão |
| Restrições | no ótimo, $\nabla f = \lambda \nabla c$; $\lambda$ costuma ter significado físico |

## Exercícios

**Conceituais (sem computador)**

1. Faça mais um passo de Newton, à mão, no exemplo da seção 4 e confira com a tabela do computador.
2. Para $f(x, y) = \tfrac12(x^2 + \kappa y^2)$, mostre que um passo de Newton leva qualquer ponto ao mínimo.
3. Classifique os pontos estacionários de $f(x, y) = x^3 - 3x + y^2$ pela hessiana. Desenhe as curvas de nível.
4. Por que, no bloco, a direção do autovalor negativo na sela é vertical? Explique pela física das molas comprimidas e desenhe a força vertical das molas em função de $y$, para $x = 1$.

**Verificação (com agente)**

5. Peça a um agente o método de Newton **amortecido**: se $H$ não for positiva definida, troque $H$ por $H + \mu I$, com $\mu$ suficiente para torná-la positiva definida, e use busca linear. Verifique, refazendo o mapa da seção 6.3, que ele nunca termina na sela.
6. Peça a um agente para encontrar o equilíbrio de uma corrente de 10 elos rígidos pendurada entre dois pontos, minimizando a energia potencial com restrições de comprimento. Verifique comparando com a catenária.
7. Peça para o agente estimar a jacobiana por diferenças finitas no método de Newton. Meça a ordem de convergência para vários valores do passo $h$ e explique o resultado com o capítulo 2.

**Projeto aberto**

8. **Projeto ótimo de uma lata.** Uma lata cilíndrica deve guardar 350 mL. Encontre o raio e a altura que minimizam a área de chapa, com e sem uma altura máxima de 12 cm. Interprete o multiplicador de Lagrange da restrição de volume e compare a resposta com latas reais.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, seção 6.6 (sistemas não lineares) e capítulos 13 a 15 (otimização).
- J. Nocedal e S. J. Wright, *Numerical Optimization*, 2ª ed., Springer, 2006, capítulos 2, 3, 6 e 12.
- S. Boyd e L. Vandenberghe, *Convex Optimization*, Cambridge University Press, 2004, disponível on-line.